# 05 — Statistical Models vs Random Walk

This notebook develops statistical challengers to the frozen Random Walk benchmark.

Models:
- ARIMA on TT levels
- ARIMA on log TT levels
- Holt damped trend
- Recent-drift forecast
- Random-Walk shrinkage

Evaluation:
- USD, GBP, EUR, JPY
- H1, H3, H7
- outer validation years 2019–2023
- 2024–2025 remains untouched

Core metrics:
- RMSE
- MAE
- Forecast Skill vs RW
- Theil's U
- Diebold–Mariano p-value

All model choices are made using earlier historical data only.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
VALIDATION_YEARS = [2019, 2020, 2021, 2022, 2023]

ARIMA_ORDERS = [
    (0, 1, 0),
    (1, 1, 0),
    (0, 1, 1),
    (1, 1, 1),
    (2, 1, 0),
    (0, 1, 2),
]

SHRINKAGE_GRID = [0.00, 0.25, 0.50, 0.75, 1.00]
REFIT_EVERY = 20

print("Configuration loaded.")

In [ ]:
def find_project_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]

    for root in candidates:
        dev_path = root / "data" / "processed" / "02_development_model_dataset.csv"
        rw_path = root / "outputs" / "predictions" / "04_random_walk_predictions_2019_2023.csv"

        if dev_path.exists() and rw_path.exists():
            return root.resolve()

    raise FileNotFoundError(
        "Required Script 02 and Script 04 outputs were not found."
    )

PROJECT_ROOT = find_project_root()

DEV_PATH = PROJECT_ROOT / "data" / "processed" / "02_development_model_dataset.csv"
RW_PATH = PROJECT_ROOT / "outputs" / "predictions" / "04_random_walk_predictions_2019_2023.csv"

TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
PRED_DIR = PROJECT_ROOT / "outputs" / "predictions"

for p in [TABLE_DIR, FIGURE_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
df = pd.read_csv(DEV_PATH, parse_dates=["Date"])

rw = pd.read_csv(
    RW_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

df = df.sort_values("Date").reset_index(drop=True)
rw = rw.sort_values(["Currency", "Horizon", "Forecast_Origin"]).reset_index(drop=True)

assert df["Date"].max() < pd.Timestamp("2024-01-01")
assert rw["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")

print("Development data:", df.shape)
print("RW rows:", len(rw))
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("PASS — final holdout is not present.")

## Core metric functions

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    if mask.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    if mask.sum() == 0:
        return np.nan

    return float(np.mean(np.abs(y_true[mask] - y_pred[mask])))


def forecast_skill(model_rmse, rw_rmse):
    if not np.isfinite(model_rmse) or not np.isfinite(rw_rmse) or rw_rmse == 0:
        return np.nan

    return float(1 - model_rmse / rw_rmse)


def theils_u(model_rmse, rw_rmse):
    if not np.isfinite(model_rmse) or not np.isfinite(rw_rmse) or rw_rmse == 0:
        return np.nan

    return float(model_rmse / rw_rmse)

In [ ]:
def diebold_mariano_test(y_true, pred_model, pred_rw, horizon=1):
    y_true = np.asarray(y_true, dtype=float)
    pred_model = np.asarray(pred_model, dtype=float)
    pred_rw = np.asarray(pred_rw, dtype=float)

    mask = np.isfinite(y_true) & np.isfinite(pred_model) & np.isfinite(pred_rw)

    y_true = y_true[mask]
    pred_model = pred_model[mask]
    pred_rw = pred_rw[mask]

    n = len(y_true)

    if n < 20:
        return np.nan, np.nan

    loss_model = (y_true - pred_model) ** 2
    loss_rw = (y_true - pred_rw) ** 2

    # Positive d means the challenger has lower squared loss than RW.
    d = loss_rw - loss_model
    d_bar = np.mean(d)

    lag = max(0, int(horizon) - 1)
    gamma0 = np.var(d, ddof=1)
    long_run_var = gamma0

    for k in range(1, lag + 1):
        if len(d) <= k:
            break

        cov_k = np.cov(d[k:], d[:-k], ddof=1)[0, 1]
        long_run_var += 2 * (1 - k / (lag + 1)) * cov_k

    if not np.isfinite(long_run_var) or long_run_var <= 0:
        return np.nan, np.nan

    dm_stat = d_bar / np.sqrt(long_run_var / n)
    p_value = 2 * student_t.sf(abs(dm_stat), df=n - 1)

    return float(dm_stat), float(p_value)

## Historical tuning design

For an outer validation year such as 2022, the immediately preceding year (2021) is used for inner model selection. Earlier observations form the inner training history.

The outer year is never used to choose the ARIMA order or shrinkage strength.

In [ ]:
def get_inner_validation_year(outer_year):
    return outer_year - 1

In [ ]:
def arima_forecast(history, order, horizon, log_transform=False):
    history = pd.Series(history, dtype=float).dropna()

    if len(history) < 80:
        return np.nan

    try:
        y = np.log(history) if log_transform else history

        trend = "t" if order[1] == 1 else "c"

        result = ARIMA(
            y,
            order=order,
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False
        ).fit()

        fc = result.forecast(steps=horizon)
        pred = float(fc.iloc[-1] if hasattr(fc, "iloc") else fc[-1])

        if log_transform:
            pred = float(np.exp(pred))

        return pred

    except Exception:
        return np.nan

In [ ]:
def holt_forecast(history, horizon):
    history = pd.Series(history, dtype=float).dropna()

    if len(history) < 80:
        return np.nan

    try:
        result = ExponentialSmoothing(
            history,
            trend="add",
            damped_trend=True,
            seasonal=None,
            initialization_method="estimated"
        ).fit(
            optimized=True,
            use_brute=False
        )

        fc = result.forecast(horizon)

        return float(fc.iloc[-1] if hasattr(fc, "iloc") else fc[-1])

    except Exception:
        return np.nan


def recent_drift_forecast(history, horizon, window):
    history = pd.Series(history, dtype=float).dropna()

    if len(history) < max(30, window + 1):
        return np.nan

    recent = history.iloc[-(window + 1):]
    avg_change = recent.diff().dropna().mean()

    return float(history.iloc[-1] + horizon * avg_change)

## Inner candidate evaluation

To keep runtime practical, tuning evaluates a regular sample of origins from the inner year.

The final reported outer-year predictions still use every Random Walk benchmark origin.

In [ ]:
def candidate_forecast(history, model_name, model_param, horizon):
    if model_name == "ARIMA_LEVEL":
        return arima_forecast(
            history,
            order=model_param,
            horizon=horizon,
            log_transform=False
        )

    if model_name == "ARIMA_LOG":
        return arima_forecast(
            history,
            order=model_param,
            horizon=horizon,
            log_transform=True
        )

    if model_name == "HOLT_DAMPED":
        return holt_forecast(
            history,
            horizon=horizon
        )

    if model_name == "RECENT_DRIFT":
        return recent_drift_forecast(
            history,
            horizon=horizon,
            window=int(model_param)
        )

    return np.nan

In [ ]:
def evaluate_candidate_on_inner(series_df, inner_year, horizon, model_name, model_param):
    inner_rows = series_df[
        series_df["Date"].dt.year == inner_year
    ].copy()

    inner_rows = inner_rows.iloc[::10].copy()

    rows = []

    for _, row in inner_rows.iterrows():
        origin_date = row["Date"]

        idx = series_df.index[
            series_df["Date"] == origin_date
        ]

        if len(idx) == 0:
            continue

        pos = int(idx[0])
        future_pos = pos + horizon

        if future_pos >= len(series_df):
            continue

        history = series_df.loc[:pos, "Value"]
        actual = float(series_df.loc[future_pos, "Value"])
        rw_pred = float(series_df.loc[pos, "Value"])

        pred = candidate_forecast(
            history,
            model_name,
            model_param,
            horizon
        )

        if np.isfinite(pred):
            rows.append({
                "Forecast_Origin": origin_date,
                "Actual": actual,
                "RW": rw_pred,
                "Model": pred
            })

    result = pd.DataFrame(rows)

    if len(result) < 15:
        return np.nan, result

    return rmse(result["Actual"], result["Model"]), result

In [ ]:
def tune_task(series_df, outer_year, horizon):
    inner_year = get_inner_validation_year(outer_year)

    if (series_df["Date"].dt.year < inner_year).sum() < 250:
        return None

    if (series_df["Date"].dt.year == inner_year).sum() < 30:
        return None

    candidates = []

    for order in ARIMA_ORDERS:
        for model_name in ["ARIMA_LEVEL", "ARIMA_LOG"]:
            score, _ = evaluate_candidate_on_inner(
                series_df,
                inner_year,
                horizon,
                model_name,
                order
            )

            candidates.append({
                "Model": model_name,
                "Parameter_Text": str(order),
                "Parameter_Object": order,
                "Inner_RMSE": score
            })

    score, _ = evaluate_candidate_on_inner(
        series_df,
        inner_year,
        horizon,
        "HOLT_DAMPED",
        None
    )

    candidates.append({
        "Model": "HOLT_DAMPED",
        "Parameter_Text": "damped_additive",
        "Parameter_Object": None,
        "Inner_RMSE": score
    })

    for window in [20, 60, 120, 250]:
        score, _ = evaluate_candidate_on_inner(
            series_df,
            inner_year,
            horizon,
            "RECENT_DRIFT",
            window
        )

        candidates.append({
            "Model": "RECENT_DRIFT",
            "Parameter_Text": str(window),
            "Parameter_Object": window,
            "Inner_RMSE": score
        })

    candidate_df = pd.DataFrame(candidates).dropna(subset=["Inner_RMSE"])

    if len(candidate_df) == 0:
        return None

    best_idx = candidate_df["Inner_RMSE"].idxmin()
    best = candidate_df.loc[best_idx]

    return {
        "Inner_Year": inner_year,
        "Best_Model": best["Model"],
        "Best_Parameter": best["Parameter_Object"],
        "Best_Parameter_Text": best["Parameter_Text"],
        "Best_Inner_RMSE": float(best["Inner_RMSE"]),
        "Candidates": candidate_df
    }

## Shrink the challenger toward Random Walk

The shrinkage forecast is:

\[
\hat P_{t+h}
=
P_t + \lambda(\hat P^{model}_{t+h}-P_t)
\]

where lambda is selected using only the inner historical year.

In [ ]:
def tune_shrinkage(series_df, inner_year, horizon, model_name, model_param):
    _, pred_df = evaluate_candidate_on_inner(
        series_df,
        inner_year,
        horizon,
        model_name,
        model_param
    )

    if len(pred_df) < 15:
        return 1.0

    scores = []

    for lam in SHRINKAGE_GRID:
        pred = pred_df["RW"] + lam * (pred_df["Model"] - pred_df["RW"])

        scores.append({
            "Lambda": lam,
            "RMSE": rmse(pred_df["Actual"], pred)
        })

    score_df = pd.DataFrame(scores)
    best = score_df.loc[score_df["RMSE"].idxmin()]

    return float(best["Lambda"])

## Generate outer-year predictions

Every reported forecast is aligned to the exact same forecast origin and actual target used by Script 04.

In [ ]:
def forecast_outer_year(currency, horizon, outer_year, model_name, model_param, shrink_lambda):
    value_col = f"{currency}_mid"

    series_df = (
        df[["Date", value_col]]
        .rename(columns={value_col: "Value"})
        .dropna()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    rw_task = rw[
        (rw["Currency"] == currency)
        & (rw["Horizon"] == horizon)
        & (rw["Forecast_Origin"].dt.year == outer_year)
    ].copy()

    rw_task = rw_task.sort_values("Forecast_Origin").reset_index(drop=True)

    rows = []

    for _, r in rw_task.iterrows():
        origin_date = r["Forecast_Origin"]

        idx = series_df.index[
            series_df["Date"] == origin_date
        ]

        if len(idx) == 0:
            continue

        pos = int(idx[0])
        history = series_df.loc[:pos, "Value"]

        raw_pred = candidate_forecast(
            history,
            model_name,
            model_param,
            horizon
        )

        rw_pred = float(r["RW_Prediction"])

        if not np.isfinite(raw_pred):
            raw_pred = rw_pred

        final_pred = rw_pred + shrink_lambda * (raw_pred - rw_pred)

        rows.append({
            "Currency": currency,
            "Horizon": horizon,
            "Validation_Year": outer_year,
            "Forecast_Origin": origin_date,
            "Target_Date": r["Target_Date"],
            "Actual": float(r["Actual"]),
            "RW_Prediction": rw_pred,
            "Raw_Stat_Prediction": raw_pred,
            "Stat_Prediction": final_pred,
            "Selected_Model": model_name,
            "Selected_Parameter": str(model_param),
            "Shrinkage_Lambda": shrink_lambda
        })

    return pd.DataFrame(rows)

## Run all currency-horizon-year tasks

This is the longest cell in the notebook because statistical models are repeatedly refitted.

Let it finish completely before moving to the result cells.

In [ ]:
all_predictions = []
selection_rows = []
candidate_tables = []

for currency in CURRENCIES:
    print("\n" + "=" * 70)
    print("Currency:", currency)
    print("=" * 70)

    series_df = (
        df[["Date", f"{currency}_mid"]]
        .rename(columns={f"{currency}_mid": "Value"})
        .dropna()
        .sort_values("Date")
        .reset_index(drop=True)
    )

    for horizon in HORIZONS:
        for outer_year in VALIDATION_YEARS:
            print(f"{currency} H{horizon} — {outer_year}")

            tuned = tune_task(
                series_df,
                outer_year,
                horizon
            )

            if tuned is None:
                selected_model = "RECENT_DRIFT"
                selected_param = 250
                inner_year = outer_year - 1
                inner_rmse = np.nan
                shrink_lambda = 0.0

            else:
                selected_model = tuned["Best_Model"]
                selected_param = tuned["Best_Parameter"]
                inner_year = tuned["Inner_Year"]
                inner_rmse = tuned["Best_Inner_RMSE"]

                tmp = tuned["Candidates"].copy()
                tmp["Currency"] = currency
                tmp["Horizon"] = horizon
                tmp["Outer_Year"] = outer_year
                candidate_tables.append(tmp)

                shrink_lambda = tune_shrinkage(
                    series_df,
                    inner_year,
                    horizon,
                    selected_model,
                    selected_param
                )

            selection_rows.append({
                "Currency": currency,
                "Horizon": horizon,
                "Outer_Year": outer_year,
                "Inner_Validation_Year": inner_year,
                "Selected_Model": selected_model,
                "Selected_Parameter": str(selected_param),
                "Selected_Inner_RMSE": inner_rmse,
                "Shrinkage_Lambda": shrink_lambda
            })

            pred = forecast_outer_year(
                currency,
                horizon,
                outer_year,
                selected_model,
                selected_param,
                shrink_lambda
            )

            all_predictions.append(pred)

stat_predictions = pd.concat(all_predictions, ignore_index=True)
model_selections = pd.DataFrame(selection_rows)

candidate_results = (
    pd.concat(candidate_tables, ignore_index=True)
    if candidate_tables
    else pd.DataFrame()
)

print("\nCompleted.")
print("Prediction rows:", len(stat_predictions))

## Year-by-year results

In [ ]:
yearly_rows = []

for (cur, h, year), g in stat_predictions.groupby(
    ["Currency", "Horizon", "Validation_Year"]
):
    stat_rmse = rmse(g["Actual"], g["Stat_Prediction"])
    stat_mae = mae(g["Actual"], g["Stat_Prediction"])
    rw_rmse = rmse(g["Actual"], g["RW_Prediction"])

    dm_stat, dm_p = diebold_mariano_test(
        g["Actual"],
        g["Stat_Prediction"],
        g["RW_Prediction"],
        horizon=h
    )

    yearly_rows.append({
        "Currency": cur,
        "Horizon": int(h),
        "Validation_Year": int(year),
        "Observations": len(g),
        "RMSE": stat_rmse,
        "MAE": stat_mae,
        "Forecast_Skill_vs_RW": forecast_skill(stat_rmse, rw_rmse),
        "Theils_U": theils_u(stat_rmse, rw_rmse),
        "DM_pvalue_vs_RW": dm_p
    })

yearly_results = pd.DataFrame(yearly_rows).sort_values(
    ["Currency", "Horizon", "Validation_Year"]
)

display(yearly_results)

yearly_results.to_csv(
    TABLE_DIR / "05_statistical_yearly_results.csv",
    index=False
)

## Pooled 2019–2023 results

This is the main table for statistical models.

In [ ]:
pooled_rows = []

for (cur, h), g in stat_predictions.groupby(["Currency", "Horizon"]):
    stat_rmse = rmse(g["Actual"], g["Stat_Prediction"])
    stat_mae = mae(g["Actual"], g["Stat_Prediction"])
    rw_rmse = rmse(g["Actual"], g["RW_Prediction"])

    dm_stat, dm_p = diebold_mariano_test(
        g["Actual"],
        g["Stat_Prediction"],
        g["RW_Prediction"],
        horizon=h
    )

    pooled_rows.append({
        "Currency": cur,
        "Horizon": int(h),
        "Observations": len(g),
        "RMSE": stat_rmse,
        "MAE": stat_mae,
        "Forecast_Skill_vs_RW": forecast_skill(stat_rmse, rw_rmse),
        "Theils_U": theils_u(stat_rmse, rw_rmse),
        "DM_pvalue_vs_RW": dm_p
    })

pooled_results = pd.DataFrame(pooled_rows).sort_values(
    ["Currency", "Horizon"]
)

display(pooled_results)

pooled_results.to_csv(
    TABLE_DIR / "05_statistical_pooled_results.csv",
    index=False
)

In [ ]:
win_table = pooled_results.copy()

win_table["Beats_RW_RMSE"] = (
    win_table["Forecast_Skill_vs_RW"] > 0
)

win_table["DM_Significant_5pct"] = (
    win_table["DM_pvalue_vs_RW"] < 0.05
)

display(
    win_table[
        [
            "Currency",
            "Horizon",
            "Forecast_Skill_vs_RW",
            "Theils_U",
            "DM_pvalue_vs_RW",
            "Beats_RW_RMSE",
            "DM_Significant_5pct"
        ]
    ]
)

print(
    "Tasks beating RW on RMSE:",
    int(win_table["Beats_RW_RMSE"].sum()),
    "/",
    len(win_table)
)

## Save predictions and model-selection evidence

In [ ]:
SELECTION_PATH = TABLE_DIR / "05_statistical_model_selections.csv"
PREDICTION_PATH = PRED_DIR / "05_statistical_predictions_2019_2023.csv"

model_selections.to_csv(SELECTION_PATH, index=False)
stat_predictions.to_csv(PREDICTION_PATH, index=False)

if len(candidate_results):
    candidate_results.to_csv(
        TABLE_DIR / "05_inner_candidate_results.csv",
        index=False
    )

print("Saved:")
print(SELECTION_PATH)
print(PREDICTION_PATH)

## RMSE comparison plots

In [ ]:
for cur in CURRENCIES:
    temp = pooled_results[
        pooled_results["Currency"] == cur
    ].copy()

    x = np.arange(len(temp))
    width = 0.35
    rw_vals = []

    for _, row in temp.iterrows():
        g = stat_predictions[
            (stat_predictions["Currency"] == cur)
            & (stat_predictions["Horizon"] == row["Horizon"])
        ]

        rw_vals.append(
            rmse(g["Actual"], g["RW_Prediction"])
        )

    plt.figure(figsize=(8, 4))

    plt.bar(
        x - width / 2,
        rw_vals,
        width,
        label="Random Walk"
    )

    plt.bar(
        x + width / 2,
        temp["RMSE"],
        width,
        label="Statistical"
    )

    plt.xticks(
        x,
        [f"H{h}" for h in temp["Horizon"]]
    )

    plt.title(f"{cur}/LKR — Statistical vs Random Walk")
    plt.ylabel("RMSE")
    plt.legend()
    plt.tight_layout()

    plt.savefig(
        FIGURE_DIR / f"05_{cur}_stat_vs_rw_rmse.png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()

## Quality gate

In [ ]:
rw_keys = set(
    zip(
        rw["Currency"],
        rw["Horizon"],
        rw["Forecast_Origin"]
    )
)

stat_keys = set(
    zip(
        stat_predictions["Currency"],
        stat_predictions["Horizon"],
        stat_predictions["Forecast_Origin"]
    )
)

quality_gate = pd.DataFrame([
    {
        "Area": "Final holdout isolation",
        "Status": "PASS",
        "Evidence": "Only pre-2024 development data used"
    },
    {
        "Area": "Frozen Random Walk benchmark",
        "Status": "PASS",
        "Evidence": "Script 04 benchmark predictions reused"
    },
    {
        "Area": "Matched forecast origins",
        "Status": "PASS" if rw_keys == stat_keys else "REVIEW",
        "Evidence": f"RW={len(rw_keys)}, Statistical={len(stat_keys)}"
    },
    {
        "Area": "Historical model selection",
        "Status": "PASS",
        "Evidence": "Previous year used for inner tuning"
    },
    {
        "Area": "Historical shrinkage selection",
        "Status": "PASS",
        "Evidence": "Lambda selected from previous-year inner validation"
    }
])

display(quality_gate)

quality_gate.to_csv(
    TABLE_DIR / "05_quality_gate.csv",
    index=False
)

In [ ]:
print("=" * 78)
print("NOTEBOOK 05 — STATISTICAL MODELS COMPLETE")
print("=" * 78)

print("\nPOOLED RESULTS:")
display(pooled_results)

print("\nMODEL SELECTIONS:")
display(model_selections)

print(
    "\nTasks beating RW on pooled RMSE:",
    int((pooled_results["Forecast_Skill_vs_RW"] > 0).sum()),
    "/",
    len(pooled_results)
)

print("\nInterpretation:")
print("Forecast Skill > 0 and Theil U < 1 = lower RMSE than Random Walk.")
print("DM p-value < 0.05 = stronger statistical evidence of different forecast loss.")
print("The 2024-2025 final holdout remains untouched.")
print("\nNext notebook: 06_Machine_Learning.ipynb")